In [ ]:
# At (Pdb) prompt:
# n next line
# s step into
# c continue
# p args / pp requested_tickers inspect vars
# l show code
# q quit
# Workflow mode
# %run -d -b 113 -b 118 -b 184 -b 195 run_one_ticker.py -- --ticker TSLA --mode workflow --model gpt-5-mini --n-times 1 --max-turns 30
# Agent mode (with MCP)
# %run -d -b 113 -b 118 -b 177 -b 186 -b 195 run_one_ticker.py -- --ticker TSLA --mode agent --model gpt-5-mini --mcp --n-times 1 --max-turns 30


In [1]:
from __future__ import annotations
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List
import os
import pandas as pd
import requests
import pandas_market_calendars as mcal


TICKERS = ["TSLA", "AMZN", "NIO", "MSFT", "AAPL", "GOOG", "NFLX", "COIN"]
output_path="/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/data/"
today_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")


In [10]:
path_tab= "/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/validation/table2_us_rows_workflow.csv"
roic_type = pd.read_csv(path_tab)
# roic_type.head()
roic_type

,ticker,analysis_date,run_id,indicator,predicted,gold,source,gold_date,capture_date,units,...,input_tokens,output_tokens,total_tokens,requests,normalised_error,abs_pct_error,normalisation_method,ratio_clip_applied,gold_min_for_scaling,gold_max_for_scaling
0,AAPL,2026-02-25,0,BVPS,5.998217e+00,5.310000e+00,gold,2026-02-25,2026-02-25,USD_per_share,...,41895,7094,48989,2,1.296077e-01,1.296077e-01,ape_fallback_zero_range,0,5.310000e+00,5.310000e+00
1,AAPL,2026-02-25,0,CashAndEquivalents,4.531700e+10,5.470000e+10,gold,2026-02-25,2026-02-25,USD,...,41895,7094,48989,2,1.715356e-01,1.715356e-01,ape_fallback_zero_range,0,5.470000e+10,5.470000e+10
2,AAPL,2026-02-25,0,CurrentRatio,9.737447e-01,8.900000e-01,gold,2026-02-25,2026-02-25,multiple,...,41895,7094,48989,2,9.409513e-02,9.409513e-02,ape_fallback_zero_range,0,8.900000e-01,8.900000e-01
3,AAPL,2026-02-25,0,EBITMargin,3.197080e+01,3.197000e+01,gold,2026-02-25,2026-02-25,percent,...,41895,7094,48989,2,2.501603e-05,2.501603e-05,ape_fallback_zero_range,0,3.197000e+01,3.197000e+01
4,AAPL,2026-02-25,0,EBIT_TTM,1.330500e+11,1.330500e+11,gold,2026-02-25,2026-02-25,USD,...,41895,7094,48989,2,1.146846e-16,1.146846e-16,ape_fallback_zero_range,0,1.330500e+11,1.330500e+11
5,AAPL,2026-02-25,0,EPS,7.460000e+00,7.460000e+00,gold,2026-02-25,2026-02-25,USD_per_share,...,41895,7094,48989,2,0.000000e+00,0.000000e+00,ape_fallback_zero_range,0,7.460000e+00,7.460000e+00
6,AAPL,2026-02-25,0,EV_EBIT,2.989571e+01,2.894000e+01,gold,2026-02-25,2026-02-25,multiple,...,41895,7094,48989,2,3.302385e-02,3.302385e-02,ape_fallback_zero_range,0,2.894000e+01,2.894000e+01
7,AAPL,2026-02-25,0,EV_EBITDA,2.747965e+01,2.660000e+01,gold,2026-02-25,2026-02-25,multiple,...,41895,7094,48989,2,3.306951e-02,3.306951e-02,ape_fallback_zero_range,0,2.660000e+01,2.660000e+01
8,AAPL,2026-02-25,0,GrossDebt,9.049700e+10,9.866000e+10,gold,2026-02-25,2026-02-25,USD,...,41895,7094,48989,2,8.273870e-02,8.273870e-02,ape_fallback_zero_range,0,9.866000e+10,9.866000e+10
9,AAPL,2026-02-25,0,GrossDebt_Equity,1.026159e+00,1.338000e+00,gold,2026-02-25,2026-02-25,multiple,...,41895,7094,48989,2,2.330647e-01,2.330647e-01,ape_fallback_zero_range,0,1.338000e+00,1.338000e+00


In [ ]:
path_tab2= "/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/validation/table2_us_summary_workflow.csv"
roic_sum = pd.read_csv(path_tab2)
roic_sum.head()

,indicator,n_points,n_rows,n_tickers,n_analysis_dates,mean_normalised_error,median_normalised_error,nmae,median_ape,penalty_rate,score_one,alpha,notes
0,EPS,1,1,1,1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,10.0,constant_gold_range_fallback
1,NetProfit_TTM,1,1,1,1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,10.0,constant_gold_range_fallback
2,EBIT_TTM,1,1,1,1,1.146846e-16,1.146846e-16,1.146846e-16,1.146846e-16,0.0,1.146846e-16,10.0,constant_gold_range_fallback
3,NetRevenue_TTM,1,1,1,1,2.402922e-06,2.402922e-06,2.402922e-06,2.402922e-06,0.0,2.402922e-06,10.0,constant_gold_range_fallback
4,EBITMargin,1,1,1,1,2.501603e-05,2.501603e-05,2.501603e-05,2.501603e-05,0.0,2.501603e-05,10.0,constant_gold_range_fallback


In [14]:
path_tab1= "/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/validation/table1_us_summary.csv"
roic_tab1 = pd.read_csv(path_tab1)
roic_tab1

,model,architecture,reflection,n_analyses,n_runs,n_buy,n_hold,n_sell,mean_nmae,mean_penalty_rate,mean_score_one,mean_score,mean_tokens_per_analysis,mean_total_tokens
0,gpt-5-mini,workflow,False,14,1,2,10,2,0.100145,0.0,0.100145,0.100145,56040.857143,56040.857143


In [6]:
split_2025 = pd.read_csv("/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/data/processed/splits/test_2025-01-01_to_2026-02-25.csv")
split_2025.head()
mask = (split_2025["date"] == "2026-02-24") & (split_2025["ticker"] == "AAPL")
check = split_2025.loc[mask].iloc[0]
check

date                            2026-02-24
ticker                                AAPL
adj_close                       272.140015
ret_1d                            0.616995
log_ret_1d                        0.022144
Volume                           -0.127226
filed                           2026-01-30
Assets                             0.77603
Liabilities                       1.737268
StockholdersEquity               -0.144499
Revenues                               NaN
NetIncomeLoss                      0.99671
OperatingIncomeLoss               1.155145
EarningsPerShareBasic            -0.028904
CommonStockSharesOutstanding       1.43771
ret_5d                            0.350847
ret_20d                           0.328203
vol_20d                          -0.575568
vol_60d                          -0.992675
target_ret_1d                      0.00768
Name: 285, dtype: object